# Repair v1: stage A with arithmetic slips

Repairer v1's candidate (D-088): stage A's recipe with the repair-only arithmetic slips added to training (a growth rate or share misstated from the right figures, the commonest natural error in the pilot). Evaluated on the same 500 tune-track items as stage A and on the tune track's arithmetic slips, and stage A is evaluated again on both with the same code, since the gate and executor changed after stage A's first evaluation (D-087, D-090, D-092). Attach the stage A notebook's output. Expect about 8.5 GPU hours.

**On Kaggle:** Create → New Notebook → File → Import Notebook → this file. Settings: *Accelerator* **GPU T4 x2**, *Internet* **on**.
For long jobs use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours and keeps everything in `/kaggle/working/outputs`.
To continue in a later notebook, add this notebook's output as input (*Add Input → Your Work*).

**Afterwards:** download `outputs/*/results.json` and `report.md` files into `runs/` in the repository, and log the session's GPU hours in `logs/gpu-hours.csv`. Checkpoints are saved regularly; re-running the same cell resumes from the latest one.

In [ ]:
# Setup: clone FaithGuard, install it with the pinned GPU packages, fetch the data.
import shutil, socket
try:
    socket.gethostbyname("pypi.org")
except OSError:
    raise RuntimeError("No internet in this session. Kaggle: verify your phone number (Settings), then switch Internet on in the right-hand panel.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU in this session. Kaggle: verify your phone number (Settings), then set Accelerator to GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
if not os.path.exists('faithguard'):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/SharafThawfeek/FaithGurd-new.git', 'faithguard'])
os.chdir('faithguard')
REQUIREMENTS = ['transformers==5.19.0', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'huggingface_hub==1.33.0', 'datasets==5.1.0', 'lettucedetect==0.2.3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
# Kaggle's image ships torchao 0.10, which PEFT 0.21 rejects when it adds LoRA to fp16 weights; nothing here uses it
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
# this kernel started before the install, and the clone's own folder would shadow the package: import from src/
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
OUT = os.path.join(WORK, 'outputs'); os.makedirs(OUT, exist_ok=True)
!faithguard data download all
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Stage A's adapter, for the comparisons

In [ ]:
import glob
ADAPTER = sorted(glob.glob('/kaggle/input/**/repair-sft/adapter', recursive=True))[0]
print(ADAPTER)
MODE = 'fp16'

## Training data with arithmetic slips (about 3 minutes on CPU)

In [ ]:
!faithguard sft --mode program --arithmetic --out runs/sft-v1

## Fine-tune (resumable)

In [ ]:
!python -m faithguard.train.repair_sft --data runs/sft-v1/repair-program-train.jsonl --out {OUT}/repair-v1 --mode $MODE

## Evaluate v1 and stage A with the same code: the tune track and its arithmetic slips

In [ ]:
!python -m faithguard.train.repair_eval --adapter {OUT}/repair-v1/adapter --load $MODE --out {OUT}/eval-v1
!python -m faithguard.train.repair_eval --adapter {OUT}/repair-v1/adapter --load $MODE --arithmetic --out {OUT}/eval-v1-arithmetic
!python -m faithguard.train.repair_eval --adapter $ADAPTER --load $MODE --out {OUT}/eval-sft-current
!python -m faithguard.train.repair_eval --adapter $ADAPTER --load $MODE --arithmetic --out {OUT}/eval-sft-arithmetic

## Results

In [ ]:
import glob, json
for path in sorted(glob.glob(OUT + '/**/results.json', recursive=True)):
    print(path)
    print(json.dumps({k: v for k, v in json.load(open(path)).items() if k in ('summary', 'channel_a', 'channel_b', 'auroc', 'false_alarm_clean', 'wrong_context_recall')}, indent=1)[:1500])